# 02 · Task 1 — the activation function

*`EncoderBlock` / `DecoderBlock` activation sweep*

Neural Audio and Speech Processing — Day 2 / Application to Speech Enhancement
Homework assignment, R. Scheibler (2026-07-02) · dataset: Voicebank-DEMAND (16 kHz)

> **Task 1.** Change the activation function in `EncoderBlock` and `DecoderBlock`.

`models/crn.py` was modified so both blocks take an `activation` argument, resolved
by `get_activation()`; the default `"leaky_relu"` (`nn.LeakyReLU(0.2)`) reproduces
the original baseline exactly, so a config file can now say
`CRNTiny(activation="prelu")` and nothing else changes.

```python
class EncoderBlock(nn.Module):
    def __init__(self, ..., activation="leaky_relu"):
        self.conv = nn.Conv2d(...)
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = get_activation(activation)
```

### The candidates and why

| Activation | Why it is worth a run |
|---|---|
| `leaky_relu` | the baseline (slope 0.2) — the control |
| `relu` | does the negative slope matter at all? |
| `prelu` | same shape, but the slope is *learned* (+7 parameters) |
| `elu` | smooth, saturating negative part — different gradient near zero |
| `gelu` | smooth, self-gating; the transformer default |
| `silu` | smooth, non-monotonic; strong in small convnets |
| `mish` | smoother SiLU variant, more expensive |

**Protocol.** All seven are trained for `SWEEP_EPOCHS` epochs with everything else
fixed (seed 42, `lr=1e-3`, batch 32), the winner is then retrained for the full
25 epochs. A short sweep is enough to rank activations because they mostly affect
early optimisation speed — but the full run is what goes in the report.

**Runtime:** 7 × ~10 min (5 epochs) + one full run ≈ 2 h on a T4.

In [ ]:
# --- Google Colab bootstrap (does nothing when running locally) --------------
# IMPORTANT: point this at the fork that contains the homework modifications
# (models/crn.py with a selectable activation, train.py with --warmup-steps).
REPO_URL = "https://github.com/Ahmed-AlGhosaini/nanoSE.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os

    if not os.path.exists("nanoSE"):
        !git clone -q $REPO_URL nanoSE
    %cd nanoSE
    !pip install -q -r requirements.txt

    import torch
    if not torch.cuda.is_available():
        print("No GPU! Runtime > Change runtime type > T4 GPU, then re-run this cell.")

In [ ]:
import sys
from pathlib import Path

# Make notebooks/nb_utils.py importable no matter where the kernel was started
for candidate in (Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent / "notebooks"):
    if (candidate / "nb_utils.py").exists():
        sys.path.insert(0, str(candidate))
        break

import matplotlib.pyplot as plt
import pandas as pd

import nb_utils

ROOT = nb_utils.bootstrap()   # chdir to the repository root + print the device

In [ ]:
SWEEP_EPOCHS = 5     # short ranking runs
FULL_EPOCHS = 25     # full-length run for the winner

CANDIDATES = ["leaky_relu", "relu", "prelu", "elu", "gelu", "silu", "mish"]

> **Resuming after a disconnect.** Every finished run is recorded in
> `notebooks/experiment_runs.json`, and the sweep loops skip anything already recorded.
> Re-running the cell after a Colab timeout continues where it stopped instead of
> starting over.

In [ ]:
activation_runs = {}
for activation in CANDIDATES:
    key = f"act_{activation}"
    done = nb_utils.recall(key)
    if done is not None:
        activation_runs[activation] = done
        print(f"[skip] {activation:<11} already trained -> {done}")
        continue

    config = nb_utils.write_config(
        f"exp_act_{activation}.py",
        name=f"act_{activation}",
        model=f'CRNTiny(activation="{activation}")',
        docstring=f"Task 1: {activation} activation in EncoderBlock/DecoderBlock.",
        epochs=SWEEP_EPOCHS,
    )
    activation_runs[activation] = nb_utils.remember(key, nb_utils.run_training(config))

## Ranking after `SWEEP_EPOCHS` epochs

In [ ]:
labels = list(activation_runs)
runs = [activation_runs[a] for a in labels]

table = nb_utils.summarize(runs, labels=labels, best_epoch=True)
table[["label", "epoch", "val_si_sdr", "pesq", "estoi", "dnsmos"]].round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
nb_utils.plot_curves(runs, labels=labels, metric="val_si_sdr", ax=axes[0],
                     title="Validation SI-SDR per epoch")

reference = float(table.loc[table.label == "leaky_relu", "val_si_sdr"].iloc[0])
nb_utils.plot_bars(list(table.label), list(table.val_si_sdr),
                   title=f"Best SI-SDR within {SWEEP_EPOCHS} epochs",
                   ylabel="Validation SI-SDR (dB)", baseline=reference, ax=axes[1])
plt.tight_layout()
plt.show()

deltas = (table.set_index("label").val_si_sdr - reference).sort_values(ascending=False)
print("SI-SDR relative to the leaky_relu baseline (dB):")
print(deltas.round(2).to_string())

In [ ]:
# PESQ tells a different story than SI-SDR often enough to be worth checking
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
nb_utils.plot_bars(list(table.label), list(table.pesq), title="PESQ", ylabel="PESQ",
                   baseline=float(table.loc[table.label == "leaky_relu", "pesq"].iloc[0]), ax=axes[0])
nb_utils.plot_bars(list(table.label), list(table.estoi), title="eSTOI", ylabel="eSTOI",
                   baseline=float(table.loc[table.label == "leaky_relu", "estoi"].iloc[0]), ax=axes[1])
plt.tight_layout()
plt.show()

## Full-length run for the winner

The activation with the highest short-run SI-SDR is retrained for the full 25
epochs so it can be compared against the baseline on equal terms.

In [ ]:
best_activation = table.iloc[0].label
print("winner of the short sweep:", best_activation)

if best_activation == "leaky_relu":
    print("The baseline activation won -- reusing the baseline run as the full-length result.")
    activation_full_run = nb_utils.recall("baseline")
else:
    activation_full_run = nb_utils.recall("activation_best_full")
    if activation_full_run is None:
        config = nb_utils.write_config(
            f"exp_act_{best_activation}_full.py",
            name=f"act_{best_activation}_full",
            model=f'CRNTiny(activation="{best_activation}")',
            docstring=f"Task 1 winner: {best_activation}, full-length run.",
            epochs=FULL_EPOCHS,
        )
        activation_full_run = nb_utils.remember("activation_best_full", nb_utils.run_training(config))

nb_utils.remember("best_activation_name", best_activation)   # picked up by notebook 05
print("full-length run:", activation_full_run)

In [ ]:
baseline_run = nb_utils.recall("baseline")
comparison = [r for r in (baseline_run, activation_full_run) if r is not None]
comparison_labels = ["baseline (leaky_relu)", f"{best_activation}"][: len(comparison)]

nb_utils.summarize(comparison, labels=comparison_labels).round(3)

In [ ]:
if len(comparison) == 2 and comparison[0] != comparison[1]:
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
    nb_utils.plot_curves(comparison, labels=comparison_labels, metric="val_si_sdr", ax=axes[0])
    nb_utils.plot_curves(comparison, labels=comparison_labels, metric="pesq", ax=axes[1])
    plt.tight_layout()
    plt.show()

## Discussion

*Fill in with your numbers.* Points worth commenting on:

* **How large is the spread?** If the whole field lands within a few tenths of a dB,
  the honest conclusion is that the activation is not the bottleneck for this model
  — and that is a perfectly good result to report.
* **Smooth vs piecewise-linear.** `gelu`/`silu`/`mish` differ from `relu` mainly in
  the gradient near zero. With BatchNorm in front of every activation the inputs are
  already centred, which is exactly where the shapes differ most.
* **`prelu`** adds only 7 parameters (one slope per block); if it wins, inspect the
  learned slopes — they say whether 0.2 was a good guess.
* **Cost.** `mish` is noticeably slower per step than `relu`; check the `Time:` column
  of the run logs before declaring a winner on quality alone.

**Next:** `03_lr_tuning.ipynb` (Task 2).